### Get all GROQ available models

In [7]:
import requests
import os
import json

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(json.dumps(response.json(), indent=2))

{
  "object": "list",
  "data": [
    {
      "id": "openai/gpt-oss-120b",
      "object": "model",
      "created": 1754408224,
      "owned_by": "OpenAI",
      "active": true,
      "context_window": 131072,
      "public_apps": null,
      "max_completion_tokens": 65536,
      "hugging_face_id": "openai/gpt-oss-120b",
      "name": "GPT OSS 120B",
      "input_modalities": [
        "text"
      ],
      "output_modalities": [
        "text"
      ],
      "context_length": 131072,
      "max_output_length": 65536,
      "pricing": {
        "prompt": "0.00000015",
        "completion": "0.0000006",
        "image": "0",
        "request": "0",
        "input_cache_read": "0.000000075"
      },
      "supported_sampling_parameters": [
        "temperature",
        "top_p",
        "stop",
        "seed",
        "max_tokens"
      ],
      "supported_features": [
        "tools",
        "json_mode",
        "structured_outputs",
        "reasoning"
      ]
    },
    {
      "id"

### Structured Output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [8]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3.6-27b", max_tokens=1000)
model

ChatGroq(output_version=None, profile={}, client=<groq.resources.chat.completions.Completions object at 0x000001CA8CE61090>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CA8CE62490>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None, max_tokens=1000)

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="the title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10 in imdb")

In [9]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={}, client=<groq.resources.chat.completions.Completions object at 0x000001CA8CE61090>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CA8CE62490>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None, max_tokens=1000), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10 in imdb', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'functi

In [10]:
response = model.invoke("provide details about the movie inception")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user asked for "details about the movie inception". This is a straightforward request for factual information about Christopher Nolan\'s 2010 film *Inception*.\n\n2.  **Identify Key Information Needed**:\n   - Title: Inception\n   - Director: Christopher Nolan\n   - Release Year: 2010\n   - Genre: Sci-fi, Action, Thriller, Heist\n   - Main Cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Cillian Murphy, Marion Cotillard, Michael Caine\n   - Plot Summary: Core premise, main conflict, key themes\n   - Themes: Reality vs. dreams, guilt, memory, subconscious, time dilation\n   - Notable Elements: Practical effects, IMAX filming, score by Hans Zimmer, ambiguous ending\n   - Reception: Critical acclaim, box office success, awards/nominations\n   - Legacy/Cultural Impact: Influence on cinema, fan theories, spawning sequels/merchandise\n\n3.  **Structure the Res

In [11]:
response = model_with_structured_output.invoke("provide details about the movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Nested structure

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Miles')], genres=['Science Fiction', 'Action', 'Thriller', 'Adventure'], budget=None)

### Examples with agent instead of chat model
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

#### Pydantic

In [13]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='edaf1d6b-2821-49c8-a968-c88e0f92aef0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User wants to extract contact info from the string: "John Doe, john@example.com, (555) 123-4567"\n   - The format provided is: Name, Email, Phone\n   - I have a tool `ContactInfo` that takes parameters: `name`, `email`, `phone` (all required).\n\n2.  **Map Input to Tool Parameters:**\n   - `name`: "John Doe"\n   - `email`: "john@example.com"\n   - `phone`: "(555) 123-4567"\n\n3.  **Check Tool Requirements:**\n   - All required parameters are present.\n   - Types match (all strings).\n\n4.  **Construct Tool Call:**\n   - Function: `ContactInfo`\n   - Arguments: `{"name": "John Doe", "email": "john@example.com", "phone": "(555) 123-4567"}`\n\n5.  **Execute Tool Call:*

In [14]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

#### Dataclasses

In [16]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')